# Geometry Submodule — Algebra-Independent Entities

**Part II · Geometric Algebra** — Tutorial 15

This tutorial covers the `pytanga.geometry` data model in full: the
algebra-independent **entity** and **operator** dataclasses, the bidirectional
`create` / `analyze` pipeline, MV-accepting constructors and factories, seeded
random generation, and the per-algebra coverage matrix.

By the end you will be able to:

- Bind any cached basis algebra to a `Geometry` and call `geo(entity)` / `geo(mv)`.
- Create and analyze the full entity set (`Point`, `Direction`, `Line`, `Plane`,
  `Circle`, `Sphere`, `PointPair`, `Space`) and operator set (`ReflectionPlane`,
  `ReflectionLine`, `ReflectionPoint`, `Inversion`, `Rotor`, `Translator`,
  `Dilator`, `Motor`, `GeneralRotor`).
- Switch OPNS / IPNS via `algebra.opns`.
- Use the plain `analyze` / `create` functions and the typed analyzers.
- Build entities from multivectors and with factory helpers.
- Generate reproducible random entities and export a snapshot.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (blades, grades). The
> viewer API is covered in [17](../17_visualizing_algebra_entities/) and in
> [Part I — Visualization](../../visualization/).

## 1. Setup

Bind the four 3D basis algebras (E3, P3, PGA3, N3). Each is served by a
precompiled C++ binding, so no toolchain is needed. The OPNS/IPNS flag is read
from the algebra (default `True`); the optional `seed` makes random generation
reproducible.

In [1]:
import math
import os

from pytanga.basis import BasisE3, BasisN3, BasisP3, BasisPGA3
from pytanga.geometry import (
    Circle, Dilator, Direction, GeneralRotor, Geometry, Inversion, Line, Motor,
    Plane, Point, PointPair, ReflectionLine, ReflectionPlane, ReflectionPoint,
    Rotor, Space, Sphere, Translator,
)

E3 = BasisE3()
N3 = BasisN3()
P3 = BasisP3()
PGA3 = BasisPGA3()

geo_e3 = Geometry(E3, seed=42)
geo_n3 = Geometry(N3, seed=42)
geo_p3 = Geometry(P3, seed=42)
geo_pga3 = Geometry(PGA3, seed=42)

## 2. The bidirectional pipeline

`geo(entity)` creates an `MV`; `geo(mv)` analyzes an `MV` back into an
entity/operator. `geo.create` / `geo.analyze` are the explicit equivalents.

In [2]:
p = Point(1, 2, 3)
mv = geo_e3(p)
print("create :", mv.to_dict())
print("analyze:", geo_e3(mv))

create : {'e1': 1.0, 'e2': 2.0, 'e3': 3.0}
analyze: Dir(1.00, 2.00, 3.00)


## 3. Entities

Entity support depends on the model. E3 sees a **Plane** (grade-2) and the
**Space** pseudoscalar; a grade-1 vector reads as a **Direction** under OPNS
(E3 cannot distinguish a point from a direction — both are grade-1). P3 and
PGA3 add **Line**; N3 adds the conformal **Circle**, **Sphere**, and
**PointPair**.

In [3]:
# E3: Plane + Space + grade-1 (reads back as Direction under OPNS).
print("E3 plane :", geo_e3.analyze(geo_e3.create(Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)))))
print("E3 space :", geo_e3.analyze(geo_e3.create(Space())))
print("E3 vector:", geo_e3.analyze(E3("3 e1 + 4 e2")))

# P3: adds Line and keeps Point distinct.
print("P3 point :", geo_p3.analyze(geo_p3.create(Point(1, 2, 3))))
print("P3 line  :", geo_p3.analyze(geo_p3.create(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)))))

# PGA3: same entity set, Gunn/Dorst grade mapping.
print("PGA3 plane:", geo_pga3.analyze(geo_pga3.create(Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1)))))
print("PGA3 point:", geo_pga3.analyze(geo_pga3.create(Point(2, -1, 5))))

# N3: the full conformal entity set.
print("N3 pointpair:", geo_n3.analyze(geo_n3.create(PointPair(point_a=Point(0, 0, 0), point_b=Point(1, 0, 0)))))
print("N3 circle   :", geo_n3.analyze(geo_n3.create(Circle(center=Point(0, 0, 0), radius=2.0, normal=Direction(0, 0, 1)))))
print("N3 sphere   :", geo_n3.analyze(geo_n3.create(Sphere(center=Point(1, 0, 0), radius=3.0))))

E3 plane : Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, -1.00))
E3 space : Space(scale=1.0)
E3 vector: Dir(3.00, 4.00, 0.00)
P3 point : Point(1.00, 2.00, 3.00)
P3 line  : Line(org=Point(-0.00, -0.00, -0.00), dir=Dir(-1.00, 0.00, 0.00))
PGA3 plane: Plane(pt=Point(0.00, 0.00, 3.00), n=Dir(0.00, 0.00, 1.00))
PGA3 point: Point(2.00, -1.00, 5.00)
N3 pointpair: PntPair(Point(0.00, 0.00, 0.00), Point(1.00, 0.00, 0.00))
N3 circle   : Circle(c=Point(0.00, 0.00, 0.00), r=2.00, n=Dir(-0.00, -0.00, 1.00))
N3 sphere   : Sphere(c=Point(1.00, -0.00, -0.00), r=3.00)


## 4. Operators

Operators are versors. `geo.which_operator` returns the typed operator (the
combined `geo.analyze` may return the equivalent entity view instead).

In [4]:
# E3: reflection and rotation.
print("E3 refl-plane:", geo_e3.which_operator(geo_e3.create(ReflectionPlane(Direction(1, 0, 0)))))
print("E3 refl-line :", geo_e3.which_operator(geo_e3.create(ReflectionLine(Direction(0, 0, 1)))))
print("E3 rotor     :", geo_e3.which_operator(geo_e3.create(Rotor(math.pi / 3, Direction(0, 0, 1)))))

# PGA3: adds point reflection, translation, and rigid motion.
print("PGA3 refl-pt :", geo_pga3.which_operator(geo_pga3.create(ReflectionPoint(Point(0, 0, 0)))))
print("PGA3 transl  :", geo_pga3.which_operator(geo_pga3.create(Translator(vector=Direction(1, 2, 0)))))
print("PGA3 genrot  :", geo_pga3.which_operator(geo_pga3.create(GeneralRotor(angle=0.5, axis=Direction(0, 0, 1), origin=Point(1, 0, 0)))))

# N3: adds inversion and dilation.
print("N3 inversion :", geo_n3.which_operator(geo_n3.create(Inversion(center=Point(0, 0, 0)))))
print("N3 dilator   :", geo_n3.which_operator(geo_n3.create(Dilator(factor=2.0))))
print("N3 motor     :", geo_n3.which_operator(geo_n3.create(Motor(rotor=Rotor(0.5, Direction(0, 0, 1)), translator=Translator(vector=Direction(1, 0, 0))))))

E3 refl-plane: ReflPlane(plane=Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(1.00, 0.00, 0.00)))
E3 refl-line : ReflLine(line=Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.00, 0.00, 1.00)))
E3 rotor     : Rotor(60.0° about Dir(0.00, 0.00, 1.00))
PGA3 refl-pt : ReflPoint(pt=Point(-0.00, -0.00, -0.00))
PGA3 transl  : Transl(Dir(1.00, 2.00, 0.00))
PGA3 genrot  : GenRotor(28.6° about Dir(0.00, 0.00, 1.00) at Point(1.00, 0.00, 0.00))
N3 inversion : Inv(c=Point(0.00, 0.00, 0.00), r=1.00)
N3 dilator   : Dilator(×2.00)
N3 motor     : GenRotor(28.6° about Dir(0.00, 0.00, 1.00) at Point(0.50, 1.96, 0.00))


## 5. OPNS / IPNS

The interpretation is an **algebra property** (`algebra.opns`, mutable).
Flipping it changes the MV produced for the same entity.

In [5]:
print("opns flag:", N3.opns)
mv_opns = geo_n3.create(Point(1, 0, 0))
N3.opns = False
mv_ipns = geo_n3.create(Point(1, 0, 0))
N3.opns = True
print("OPNS point:", mv_opns.to_dict())
print("IPNS point:", mv_ipns.to_dict())

opns flag: True
OPNS point: {'e1': 1.0, 'e5': 1.0}
IPNS point: {'e1234': 1.0, 'e2345': -1.0}


## 6. Plain functions and typed analyzers

`analyze()` / `create()` (plus `_entity` / `_operator` variants) are the
stateless alternative. The per-entity analyzers live in
`pytanga.geometry.analysis`.

In [6]:
from pytanga.geometry import analyze, analyze_entity, analyze_operator, create
from pytanga.geometry.analysis import analyze_sphere

mv = create(E3, ReflectionPlane(Direction(0, 1, 0)))
print("plain create+analyze:", analyze(mv))

sph_mv = geo_n3.create(Sphere(center=Point(1, 0, 0), radius=3.0))
print("analyze_entity       :", analyze_entity(sph_mv))
print("analyze_sphere       :", analyze_sphere(sph_mv))
print("analyze_operator     :", analyze_operator(geo_n3.create(Inversion(center=Point(0, 0, 0)))))

plain create+analyze: Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, -1.00, 0.00))
analyze_entity       : Sphere(c=Point(1.00, -0.00, -0.00), r=3.00)
analyze_sphere       : Sphere(c=Point(1.00, -0.00, -0.00), r=3.00)
analyze_operator     : Inv(c=Point(0.00, 0.00, 0.00), r=1.00)


## 7. MV-accepting constructors and factories

Every entity dataclass also accepts an `MV` (e.g. `Point(mv)`), and factory
helpers coerce their geometric arguments the same way.

In [7]:
print("Point(mv)      :", Point(E3("3 e1 + 4 e2 + 5 e3")))
print("Direction(mv)  :", Direction(E3("e1 + 2 e2 + 3 e3")))
print("Line.from_points:", Line.from_points(Point(0, 0, 0), Point(1, 0, 0)))
print("Plane.from_corner_and_span:", Plane.from_corner_and_span(Point(0, 0, 0), Direction(1, 0, 0), Direction(0, 1, 0)))
print("Sphere(mv)     :", Sphere(geo_n3.create(Sphere(center=Point(1, 0, 0), radius=3.0))))
print("PointPair(mv)  :", PointPair(geo_n3.create(PointPair(point_a=Point(0, 0, 0), point_b=Point(1, 0, 0)))))

Point(mv)      : Point(3.00, 4.00, 5.00)
Direction(mv)  : Dir(1.00, 2.00, 3.00)
Line.from_points: Line(org=Point(0.00, 0.00, 0.00), dir=Dir(1.00, 0.00, 0.00))
Plane.from_corner_and_span: Plane(pt=Point(0.50, 0.50, 0.00), n=Dir(0.00, 0.00, 1.00))
Sphere(mv)     : Sphere(c=Point(1.00, -0.00, -0.00), r=3.00)
PointPair(mv)  : PntPair(Point(0.00, 0.00, 0.00), Point(1.00, 0.00, 0.00))


## 8. Random generation

`RndPoint` / `RndDirection` are lazy generators materialized by the `Geometry`
RNG. Tuple specs are uniform ranges; use `Uniform` / `Normal`, and `count=` to
draw a list.

In [8]:
from pytanga.geometry import Normal, RndDirection, RndPoint, Uniform

print("direction    :", geo_e3(RndDirection((-1, 1), (-1, 1), (-1, 1))))
print("point        :", geo_e3(RndPoint(Normal(0, 1), Uniform(-1, 1), Normal(2, 0.1))))
print("point count=3:", [m.to_dict() for m in geo_e3(RndPoint(count=3))])

direction    : 0.5479 e1 - 0.1222 e2 + 0.7172 e3
point        : 0.9406 e1 - 0.8116 e2 + 1.87 e3
point count=3: [{'e1': 0.5222794039807059, 'e2': 0.5721286105539076, 'e3': -0.7437727346489083}, {'e1': -0.09922812420886573, 'e2': -0.25840395153483753, 'e3': 0.8535299776972036}, {'e1': 0.2877302401613291, 'e2': 0.64552322654166, 'e3': -0.11317160234533774}]


## 9. Algebra independence — the eight basis classes

The eight basis classes collapse to five cached `(dim, sig)` signatures:
E2=`(2,0)`, E3/P2=`(3,0)`, P3=`(4,0)`, N2/PGA2=`(4,8)`, N3/PGA3=`(5,16)`. The
same dataclasses work unchanged; 2D models keep `z = 0`.

In [9]:
from pytanga.basis import BasisE2, BasisN2, BasisP2, BasisPGA2

for cls in (BasisE2, BasisE3, BasisP2, BasisP3, BasisN2, BasisN3, BasisPGA2, BasisPGA3):
    a = cls()
    print(f"{cls.__name__:10s} dim={a.dim} sig={a.sig}")

geo_e2 = Geometry(BasisE2())
print("E2 point:", geo_e2.analyze(geo_e2.create(Point(3, 4, 0))))
print("E2 rotor:", geo_e2.which_operator(geo_e2.create(Rotor(1.0, Direction(0, 0, 1)))))

BasisE2    dim=2 sig=0
BasisE3    dim=3 sig=0
BasisP2    dim=3 sig=0
BasisP3    dim=4 sig=0
BasisN2    dim=4 sig=8
BasisN3    dim=5 sig=16
BasisPGA2  dim=4 sig=8
BasisPGA3  dim=5 sig=16


E2 point:

 Dir(3.00, 4.00, 0.00)
E2 rotor: Rotor(57.3° about Dir(0.00, 0.00, 1.00))


## 10. Round-trip coverage matrix

`create` then `analyze` (entities) / `which_operator` (operators) round-trips
each dataclass. A `~` marks an expected alias: E3 cannot distinguish `Point`
from `Direction` (both grade-1), and a `Motor` analyzes as a `GeneralRotor`
(its screw-axis form).

In [10]:
def roundtrip(geo, label, entities, operators):
    print(f"-- {label} --")
    for e in entities:
        r = geo.analyze(geo.create(e))
        mark = "  " if type(r) is type(e) else "~ "
        print(f"  {mark}{type(e).__name__:14s} -> {type(r).__name__:14s} {r}")
    for op in operators:
        r = geo.which_operator(geo.create(op))
        mark = "  " if type(r) is type(op) else "~ "
        print(f"  {mark}{type(op).__name__:14s} -> {type(r).__name__:14s} {r}")

roundtrip(geo_e3, "E3",
    [Point(1, 2, 3), Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)), Space()],
    [ReflectionPlane(Direction(0, 0, 1)), ReflectionLine(Direction(0, 0, 1)), Rotor(math.pi / 3, Direction(0, 0, 1))])

roundtrip(geo_p3, "P3",
    [Point(1, 2, 3), Direction(1, 1, 1), Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)),
     Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)), Space()],
    [ReflectionPlane(Direction(0, 0, 1)), ReflectionLine(Direction(0, 0, 1)), Rotor(math.pi / 4, Direction(0, 0, 1))])

roundtrip(geo_pga3, "PGA3",
    [Point(2, -1, 5), Direction(1, 0, 0), Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)),
     Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1)), Space()],
    [ReflectionPlane(Direction(0, 0, 1)), ReflectionLine(Direction(0, 0, 1)), ReflectionPoint(Point(0, 0, 0)),
     Rotor(math.pi / 4, Direction(0, 0, 1)), Translator(vector=Direction(1, 2, 0)),
     Motor(rotor=Rotor(0.5, Direction(0, 0, 1)), translator=Translator(vector=Direction(1, 0, 0))),
     GeneralRotor(angle=0.5, axis=Direction(0, 0, 1), origin=Point(1, 0, 0))])

roundtrip(geo_n3, "N3",
    [Point(1, 2, 3), Direction(1, 0, 0), PointPair(point_a=Point(0, 0, 0), point_b=Point(1, 0, 0)),
     Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), Circle(center=Point(0, 0, 0), radius=2.0, normal=Direction(0, 0, 1)),
     Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)), Sphere(center=Point(1, 0, 0), radius=3.0), Space()],
    [ReflectionPlane(Direction(0, 0, 1)), Inversion(center=Point(0, 0, 0)), Rotor(math.pi / 4, Direction(0, 0, 1)),
     Translator(vector=Direction(1, 2, 0)), Dilator(factor=2.0),
     Motor(rotor=Rotor(0.5, Direction(0, 0, 1)), translator=Translator(vector=Direction(1, 0, 0))),
     GeneralRotor(angle=0.5, axis=Direction(0, 0, 1), origin=Point(1, 0, 0))])

-- E3 --
  ~ Point          -> Direction      Dir(1.00, 2.00, 3.00)
    Plane          -> Plane          Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, -1.00))
    Space          -> Space          Space(scale=1.0)
    ReflectionPlane -> ReflectionPlane ReflPlane(plane=Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, 1.00)))
    ReflectionLine -> ReflectionLine ReflLine(line=Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.00, 0.00, 1.00)))
    Rotor          -> Rotor          Rotor(60.0° about Dir(0.00, 0.00, 1.00))
-- P3 --
    Point          -> Point          Point(1.00, 2.00, 3.00)
    Direction      -> Direction      Dir(1.00, 1.00, 1.00)
    Line           -> Line           Line(org=Point(-0.00, -0.00, -0.00), dir=Dir(-1.00, 0.00, 0.00))
    Plane          -> Plane          Plane(pt=Point(-0.00, -0.00, -0.00), n=Dir(0.00, 0.00, 1.00))
    Space          -> Space          Space(scale=1.0)
    ReflectionPlane -> ReflectionPlane ReflPlane(plane=Plane(pt=Point(0.00, 0.00, 0.00), n=

## 11. Visual example

Feed the geometry-created multivectors into `pytanga.viz.Visualizer`; raw
multivectors are analyzed on the way in. The viewer API itself is covered in
[17](../17_visualizing_algebra_entities/) and [Part I — Visualization](../../visualization/).

In [ ]:
from pytanga.viz import Visualizer

plane_mv = geo_e3.create(Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)))
rotor_mv = geo_e3.create(Rotor(math.pi / 4, Direction(0, 0, 1)))
sphere_mv = geo_n3.create(Sphere(center=Point(1, 0, 0), radius=1.5))

viz = Visualizer(title="E3 + N3 — plane, rotor, sphere")
viz.add(plane_mv, color="#44ff44", opacity=0.35, label="plane (E3)")
viz.add(rotor_mv, color="#ffcc00", label="rotor (E3)")
viz.add(sphere_mv, color="#4488ff", opacity=0.5, label="sphere (N3)")

viz.display_snapshot()

## 12. Summary & next steps

| Task | API |
|---|---|
| Bind an algebra | `geo = Geometry(E3, seed=42)` |
| Entity/Operator → MV | `geo(entity)`, `geo.create(entity)` |
| MV → Entity/Operator | `geo(mv)`, `geo.analyze(mv)` |
| Typed analysis | `geo.which_entity(mv)`, `geo.which_operator(mv)` |
| OPNS / IPNS | `algebra.opns` (mutable) |
| Plain functions | `analyze(mv)`, `create(algebra, entity)` |
| Typed analyzers | `pytanga.geometry.analysis.analyze_sphere` etc. |
| MV constructor | `Point(mv)`, `Sphere(mv)` |
| Factories | `Line.from_points`, `Plane.from_corner_and_span` |
| Random entities | `RndPoint`, `RndDirection`, `Uniform`, `Normal` |

**Where to go next:**

- [**16 · Custom Algebras**](../16_custom_algebras/) — constructing algebras
  beyond the built-in basis classes.
- [**17 · Visualizing Algebra Entities**](../17_visualizing_algebra_entities/) —
  mapping the full entity/operator set into the viewer.